# Update New Mexico data center loadsite input

This notebook updates the New Mexico data center loadsite input using EPRI Powering Intelligence data: https://powering-intelligence.epri.com/dashboard/

Assumptions:
- Use the **Medium** scenario.
- Use EPRI annual energy values for **2026–2030**.
- Estimate **2031–2032** by applying the average annual increase from 2026 to 2030.
- Convert annual energy to a flat hourly load in MW.

Formula:

$$\text{Flat MW} = \frac{\text{Annual Energy TWh} \times 1{,}000{,}000}{8{,}760}$$

The 2031 and 2032 values are not EPRI forecasts. They are a simple linear extrapolation for modeling purposes and are uncertain.

## 1. Setup


In [ ]:
from pathlib import Path
import pandas as pd

REPO_ROOT = Path.cwd().resolve()
while not (REPO_ROOT / "runreeds.py").exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent

EPRI_CSV_PATH = REPO_ROOT / "CEPM/preprocessing/dc_load_nm/epri_powering_intelligence_nm.csv"
OUTPUT_CSV = REPO_ROOT / "inputs/load/loadsite_st_NM_EPRI_medium_2026_2032.csv"

SCENARIO = 'Medium'
STATE = 'NM'
HOURS_PER_YEAR = 8760

EPRI_YEARS = [2026, 2027, 2028, 2029, 2030]
EXTRAPOLATED_YEARS = [2031, 2032]

## 2. Read EPRI data

In [ ]:
epri = pd.read_csv(EPRI_CSV_PATH)

# Clean basic column types
epri['Scenario'] = epri['Scenario'].astype(str).str.strip()
epri['State'] = epri['State'].astype(str).str.strip()
epri['Year'] = pd.to_numeric(epri['Year'], errors='raise').astype(int)
epri['Annual Energy (TWh)'] = pd.to_numeric(epri['Annual Energy (TWh)'], errors='raise')

epri.head()


## 3. Keep Medium scenario for NM


In [ ]:
medium_nm = epri[(epri['Scenario'] == SCENARIO) & (epri['State'] == STATE)].copy()
medium_nm = medium_nm[['State', 'Year', 'Annual Energy (TWh)']]

medium_nm


## 4. Extrapolate 2031 and 2032

EPRI values are available through 2030 in this dataset. For 2031 and 2032, this notebook uses the average annual increase from 2026 to 2030:

$$\frac{\text{Energy}_{2030} - \text{Energy}_{2026}}{2030 - 2026}$$


In [ ]:
energy_2026 = medium_nm.loc[medium_nm['Year'] == 2026, 'Annual Energy (TWh)'].iloc[0]
energy_2030 = medium_nm.loc[medium_nm['Year'] == 2030, 'Annual Energy (TWh)'].iloc[0]

annual_growth_twh = (energy_2030 - energy_2026) / (2030 - 2026)

print(f'2026 annual energy: {energy_2026:.3f} TWh')
print(f'2030 annual energy: {energy_2030:.3f} TWh')
print(f'Average annual increase: {annual_growth_twh:.3f} TWh/year')


In [ ]:
extrapolated_rows = []

for year in EXTRAPOLATED_YEARS:
    annual_energy = energy_2030 + (year - 2030) * annual_growth_twh
    extrapolated_rows.append({
        'State': STATE,
        'Year': year,
        'Annual Energy (TWh)': annual_energy,
    })

extrapolated = pd.DataFrame(extrapolated_rows)
extrapolated


## 5. Create 2026–2032 loadsite table


In [ ]:
# Use EPRI values for 2026-2030
epri_2026_2030 = medium_nm[medium_nm['Year'].isin(EPRI_YEARS)]

# Add extrapolated values for 2031-2032
loadsite = pd.concat([epri_2026_2030, extrapolated], ignore_index=True)

# Convert annual energy to flat MW
loadsite['MW'] = (loadsite['Annual Energy (TWh)'] * 1_000_000 / HOURS_PER_YEAR).round().astype(int)

# Keep only the columns required by the loadsite file
loadsite = loadsite[['State', 'Year', 'MW']]
loadsite.columns = ['loadsitereg', 't', 'MW']

loadsite


## 6. Write the CSV


In [ ]:
csv_text = '*loadsitereg,t,MW\n' + loadsite.to_csv(index=False, header=False)

print(csv_text)

OUTPUT_CSV.write_text(csv_text)

print(f'Created {OUTPUT_CSV}')
